In [1]:
pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import librosa
import scipy.signal as signal
import pandas as pd

# -----------------------------------------
# Utility Functions
# -----------------------------------------

def segment_audio(waveform, n_segments=3):
    L = len(waveform)
    split = np.array_split(waveform, n_segments)
    return split

def compute_slope(values):
    """Slope across early-mid-late = relaxation trend."""
    x = np.array([0, 1, 2])   # segment indices
    y = np.array(values)
    slope = np.polyfit(x, y, 1)[0]
    return slope

def hilbert_envelope(w):
    analytic = signal.hilbert(w)
    envelope = np.abs(analytic)
    return np.mean(envelope)

# -----------------------------------------
# Main TSRP + Calmness Metrics Function
# -----------------------------------------

def compute_TSRP_features(waveform, sr):

    segments = segment_audio(waveform, 3)

    # ----------- Base TSRP Features -----------
    rms_vals   = [librosa.feature.rms(y=s)[0].mean() for s in segments]
    zcr_vals   = [librosa.feature.zero_crossing_rate(y=s)[0].mean() for s in segments]

    rms_slope = compute_slope(rms_vals)
    zcr_slope = compute_slope(zcr_vals)

    # ----------- Additional Calmness Metrics -----------

    # 1. Temporal Smoothness Index (TSI) – low variation = calmer
    tsi_vals = [np.std(s) for s in segments]
    tsi_slope = compute_slope(tsi_vals)

    # 2. Spectral Stability Index (SSI) – centroid stability
    centroid_vals = [librosa.feature.spectral_centroid(y=s, sr=sr)[0].mean() for s in segments]
    ssi_slope = compute_slope(centroid_vals)

    # 3. Hilbert Envelope Decay Rate (HEDR)
    hilbert_vals = [hilbert_envelope(s) for s in segments]
    hedr_slope = compute_slope(hilbert_vals)

    return {
        # TSRP
        "RMS_Early": rms_vals[0], "RMS_Mid": rms_vals[1], "RMS_Late": rms_vals[2],
        "RMS_Slope": rms_slope,

        "ZCR_Early": zcr_vals[0], "ZCR_Mid": zcr_vals[1], "ZCR_Late": zcr_vals[2],
        "ZCR_Slope": zcr_slope,

        # New Calmness Indicators
        "TSI_Early": tsi_vals[0], "TSI_Mid": tsi_vals[1], "TSI_Late": tsi_vals[2],
        "TSI_Slope": tsi_slope,

        "SSI_Early": centroid_vals[0], "SSI_Mid": centroid_vals[1], "SSI_Late": centroid_vals[2],
        "SSI_Slope": ssi_slope,

        "HEDR_Early": hilbert_vals[0], "HEDR_Mid": hilbert_vals[1], "HEDR_Late": hilbert_vals[2],
        "HEDR_Slope": hedr_slope,
    }

# -----------------------------------------
# Apply to Full Dataset & Save to Excel
# -----------------------------------------

root_dir = "/kaggle/input/qmsat-dataset/ATS-data"
records = []

class_map = {"Music": 0, "Normal(Silence)": 1, "Tilawat-e-QuranPak": 2}

import os
for cls in os.listdir(root_dir):
    cls_path = os.path.join(root_dir, cls)
    if not os.path.isdir(cls_path): continue
    label = class_map.get(cls, None)
    if label is None: continue

    for file in os.listdir(cls_path):
        if file.endswith(".wav"):
            fp = os.path.join(cls_path, file)
            waveform, sr = librosa.load(fp, sr=16000)

            feats = compute_TSRP_features(waveform, sr)
            feats["Class"] = cls
            feats["File"] = file
            records.append(feats)

df = pd.DataFrame(records)

# Save to excel
df.to_excel("/kaggle/working/TSRP_and_Calmness_Features.xlsx", index=False)

df.head()


,RMS_Early,RMS_Mid,RMS_Late,RMS_Slope,ZCR_Early,ZCR_Mid,ZCR_Late,ZCR_Slope,TSI_Early,TSI_Mid,...,SSI_Early,SSI_Mid,SSI_Late,SSI_Slope,HEDR_Early,HEDR_Mid,HEDR_Late,HEDR_Slope,Class,File
0,0.013135,0.023846,0.025777,0.006321,0.362121,0.324360,0.305077,-0.028522,0.013478,0.034059,...,2077.099918,2190.673426,1743.218286,-166.940816,0.013514,0.022949,0.023556,0.005021,Music,13_music_f_3.wav
1,0.132078,0.130934,0.132474,0.000198,0.080678,0.059468,0.060401,-0.010138,0.134201,0.132933,...,497.784965,488.117700,486.261566,-5.761700,0.129948,0.136756,0.136558,0.003305,Music,5_music_m_13.wav
2,0.040615,0.038929,0.038808,-0.000904,0.295162,0.302122,0.310380,0.007609,0.042839,0.040566,...,2730.146451,2866.436129,2919.375168,94.614358,0.047577,0.046070,0.046517,-0.000530,Music,18_music_f_35.wav
3,0.007652,0.006557,0.005938,-0.000857,0.394570,0.362857,0.352548,-0.021011,0.008703,0.005026,...,2120.317994,2046.993713,2006.486395,-56.915800,0.006911,0.006494,0.006308,-0.000301,Music,3_music_f_20.wav
4,0.063125,0.154094,0.095174,0.016024,0.289234,0.315742,0.295496,0.003131,0.075630,0.192081,...,2822.958229,3064.461631,2834.416966,5.729369,0.073316,0.172373,0.106289,0.016486,Music,15_music_f_38.wav
